# dataflow1

Converted from an Azure Data Factory / Synapse **mapping data flow** (Data Flow Script)
to a Microsoft Fabric PySpark notebook by the `mdf-to-fabric-spark` skill.

- **Source DFS:** `adf_exports/dataflow1.txt`
- Each cell corresponds to one MDF transformation; the DataFrame variable name matches the MDF stream name.
- **DAG:** `source1` (delimited / CSV read) → `sink1` (write).
- Review items are listed in the accompanying `dataflow1_conversion_report.md`.

In [ ]:
# --- Parameters (the MDF script has no parameters{} block; these expose the literal
#     dataset properties so the notebook is configurable / overridable at run time) ---

# ---- Source (delimited / CSV): MDF source1 ----
SOURCE_CONTAINER   = "msexports"   # MDF: fileSystem
SOURCE_FOLDER_PATH = "focuscost/focus/must-focus-cost/20260401-20260430/8b451d18-4a38-4be1-b8a9-64fa666d531d"  # MDF: folderPath
SOURCE_FILE_NAME   = "manifest.json"  # MDF: fileName

# MDF delimited options
COLUMN_DELIMITER       = ","    # MDF: columnDelimiter
QUOTE_CHAR             = '"'    # MDF: quoteChar
ESCAPE_CHAR            = "\\"    # MDF: escapeChar (a single backslash)
COLUMN_NAMES_AS_HEADER = False  # MDF: columnNamesAsHeader (False -> columns _c0, _c1, ...)

# ---- Sink: MDF sink1 (inline / dataset-ref: format & path are NOT present in the DFS) ----
# TODO: set these to the original sink destination. See the conversion report.
SINK_FORMAT     = "delta"        # TODO confirm original sink format (delta/parquet/csv)
SINK_CONTAINER  = "TODO_set_me"  # TODO original sink filesystem/container (if path-based)
SINK_PATH       = "TODO_set_me"  # TODO original sink folderPath, or a lakehouse table name for delta
SINK_WRITE_MODE = "overwrite"    # TODO confirm write behavior (overwrite/append)

# --- Storage / Fabric config (set for your environment; do NOT hardcode secrets) ---
STORAGE_ACCOUNT = "TODO_set_me"  # ADLS Gen2 account hosting the containers above
USE_ONELAKE     = False          # True to read/write via the attached Fabric lakehouse Files area

In [ ]:
# --- Imports (the spark session already exists in a Fabric notebook) ---
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *


def storage_base(container):
    """Build the base path for a source/sink container."""
    if USE_ONELAKE:
        # Reading/writing via the attached lakehouse Files area:
        return "/lakehouse/default/Files"
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

## Sources

**`source1`** — delimited (CSV) source.

```
source(useSchema: false, allowSchemaDrift: true, validateSchema: false,
  ignoreNoFilesFound: false, format: 'delimited', fileSystem: 'msexports',
  folderPath: 'focuscost/focus/must-focus-cost/20260401-20260430/8b451d18-4a38-4be1-b8a9-64fa666d531d',
  fileName: 'manifest.json', columnDelimiter: ',', escapeChar: '\\', quoteChar: '"',
  columnNamesAsHeader: false) ~> source1
```

⚠️ The MDF reads **`manifest.json` as *delimited* text** (not JSON) with no header, so Spark
produces positional columns `_c0`, `_c1`, … split on `,`. `allowSchemaDrift: true` means no
fixed projection is pinned. Verify this positional read is intended (see conversion report).

In [ ]:
# source1: delimited read of a single file (fileName pins the exact object).
source1 = (
    spark.read.format("csv")
    .option("header", COLUMN_NAMES_AS_HEADER)   # columnNamesAsHeader: false
    .option("sep", COLUMN_DELIMITER)            # columnDelimiter: ','
    .option("quote", QUOTE_CHAR)                # quoteChar: '"'
    .option("escape", ESCAPE_CHAR)              # escapeChar: '\\'
    .option("inferSchema", False)               # MDF delimited w/o output(): all columns as string
    .load(f"{storage_base(SOURCE_CONTAINER)}/{SOURCE_FOLDER_PATH}/{SOURCE_FILE_NAME}")
)

## Sinks

**`sink1`** — writes `source1`.

```
source1 sink(allowSchemaDrift: true, validateSchema: false,
  skipDuplicateMapInputs: true, skipDuplicateMapOutputs: true) ~> sink1
```

⚠️ The sink has **no `format` / `fileSystem` / `folderPath`** in the DFS (inline or
dataset-reference sink), so the destination is unknown. Set the `SINK_*` values in the
parameters cell before running (see conversion report).

In [ ]:
# sink1: destination not specified in the DFS — configure SINK_* in the parameters cell.
if "TODO_set_me" in (SINK_CONTAINER, SINK_PATH, STORAGE_ACCOUNT):
    raise ValueError(
        "Sink destination is not set. The MDF sink1 has no format/fileSystem/folderPath; "
        "set SINK_FORMAT / SINK_CONTAINER / SINK_PATH (and STORAGE_ACCOUNT) in the parameters cell."
    )

writer = source1.write.format(SINK_FORMAT).mode(SINK_WRITE_MODE)
if SINK_FORMAT == "delta" and "/" not in SINK_PATH:
    writer.saveAsTable(SINK_PATH)            # managed Delta table (SINK_PATH is a table name)
else:
    writer.save(f"{storage_base(SINK_CONTAINER)}/{SINK_PATH}")

## Validation (optional)

In [ ]:
# Schema / row-count sanity checks — see references/validation-and-testing.md
print("source1 rows:", source1.count())
source1.printSchema()
source1.show(5, truncate=False)